In [109]:
import pandas as pd
import stock_load as loader
import numpy as np
from backtesting import Backtest, Strategy

In [110]:
loader.download_data(loader.TICKERS_70,start_date = '2015-01-01',end_date='2025-10-01')

------- Quá trình tải dữ liệu (Fresh Download) ----------------
⬇️ Đang tải dữ liệu từ 2015-01-01 đến 2025-10-01...


[*********************100%***********************]  70 of 70 completed
d:\Computaional_Finance\stock_load.py:63: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_stacked = raw_data.stack(level=0)


💾 Đã lưu dữ liệu mới vào 'data/stock_data_70.csv'.
-------------------- Hoàn thành -----------------


Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2015-01-02 AAPL     27.847500   27.860001   26.837500   27.332500   24.237556   
           ABBV     65.440002   66.400002   65.440002   65.889999   42.091431   
           ADBE     72.699997   73.199997   71.889999   72.339996   72.339996   
           AEP      60.880001   61.240002   60.389999   61.150002   41.371323   
           AMD       2.670000    2.670000    2.670000    2.670000    2.670000   
...                       ...         ...         ...         ...         ...   
2025-09-30 UNH     343.750000  349.320007  342.329987  345.299988  342.993866   
           UNP     237.000000  237.000000  234.820007  236.369995  234.995346   
           V       339.820007  345.609985  338.510010  341.380005  340.705139   
           WMT     103.000000  103.940002  102.720001  103.059998  102.850342   
           XOM     113.349998  113.489998  111.940002  112.750000  111.772377   

                      Volume  
Date       Ticker             
2015-01-02 AAPL    212818400  
           ABBV      5086100  
           ADBE      2349200  
           AEP       2007400  
           AMD             0  
...                      ...  
2025-09-30 UNH       7429700  
           UNP       3797500  
           V         8164400  
           WMT      13873400  
           XOM      18076200  

[189140 rows x 6 columns]

In [135]:
stock_70_data = loader.load_local_data()

📂 Đang đọc dữ liệu từ đĩa: data/stock_data_70.csv...
✅ Đã đọc dữ liệu thành công.


In [136]:
stock_70_data.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 189140 entries, (Timestamp('2015-01-02 00:00:00'), 'AAPL') to (Timestamp('2025-09-30 00:00:00'), 'XOM')
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Open       189140 non-null  float64
 1   High       189140 non-null  float64
 2   Low        189140 non-null  float64
 3   Close      189140 non-null  float64
 4   Adj Close  189140 non-null  float64
 5   Volume     189140 non-null  int64  
dtypes: float64(5), int64(1)
memory usage: 9.3+ MB


In [128]:
# 1. Tính indicator 
# Hàm tính cho 1 mã
# Hàm tính cho nhiều mã 
def generate_bollinger_band_indicator(df_in, N = 20, K = 2):
    df_out = df_in.copy()
    price = df_out['Close']
    votatility = price.rolling(N).std()
    
    df_out["Middle"] = price.rolling(N).mean()
    df_out["Upper"] = df_out["Middle"] + K *  votatility
    df_out["Lower"] = df_out["Middle"] - K * votatility
    df_out["Zscore"] = (price - df_out["Middle"]) / votatility
    df_out["PercentB"] = (price - df_out["Lower"]) / (df_out["Upper"] - df_out["Lower"])

    return df_out 

def full_generate_bollinger_band_indicator(df):
    df_processed = df.groupby('Ticker', group_keys = False).apply(generate_bollinger_band_indicator)
    return df_processed


In [114]:
processed_stock_70_data = full_generate_bollinger_band_indicator(stock_70_data)
processed_stock_70_data.tail()

Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2025-09-30 UNH     343.750000  349.320007  342.329987  345.299988  342.993866   
           UNP     237.000000  237.000000  234.820007  236.369995  234.995346   
           V       339.820007  345.609985  338.510010  341.380005  340.705139   
           WMT     103.000000  103.940002  102.720001  103.059998  102.850342   
           XOM     113.349998  113.489998  111.940002  112.750000  111.772377   

                     Volume      Middle       Upper       Lower    Zscore  \
Date       Ticker                                                           
2025-09-30 UNH      7429700  338.821500  366.854815  310.788184  0.462199   
           UNP      3797500  222.945499  237.992711  207.898286  1.784317   
           V        8164400  341.652995  350.040309  333.265682 -0.065096   
           WMT     13873400  102.486501  104.975474   99.997527  0.460830   
           XOM     18076200  113.011000  116.921820  109.100180 -0.133476   

                   PercentB  
Date       Ticker            
2025-09-30 UNH     0.615550  
           UNP     0.946079  
           V       0.483726  
           WMT     0.615208  
           XOM     0.466631

In [129]:
#2. Tạo tính hiệu 
def generate_bollinger_signal(df):
    df_out = df.copy()

    # Điều kiện vào lệnh
    df_out["Long_entry"]  = (df_out["Close"] < df_out["Lower"]) | ((df_out["Zscore"] < -2) & (df_out["PercentB"] < 0))
    df_out["Short_entry"] = (df_out["Close"] > df_out["Upper"]) | ((df_out["Zscore"] > 2) & (df_out["PercentB"] > 1))

    # Điều kiện thoát lệnh
    df_out["Long_exit"]  = df_out["Close"] > df_out["Middle"]
    df_out["Short_exit"] = df_out["Close"] < df_out["Middle"]

    # Tạo cột signal
    df_out["Signal"] = 0

    position = 0  # 1 = long, -1 = short, 0 = neutral
    signals = []

    for i in range(len(df_out)):
        if position == 0:
            if df_out["Long_entry"].iloc[i]:
                position = 1
            elif df_out["Short_entry"].iloc[i]:
                position = -1

        elif position == 1:
            if df_out["Long_exit"].iloc[i]:
                position = 0

        elif position == -1:
            if df_out["Short_exit"].iloc[i]:
                position = 0

        signals.append(position)

    df_out["Signal"] = signals
    return df_out

def full_generate_bollinger_signal(df):
    df_processed = df.groupby("Ticker", group_keys=False).apply(generate_bollinger_signal)
    return df_processed

In [116]:
full_generate_bollinger_signal(processed_stock_70_data)

,,Open,High,Low,Close,Adj Close,Volume,Middle,Upper,Lower,Zscore,PercentB,Long_entry,Short_entry,Long_exit,Short_exit,Signal
Date,Ticker,,,,,,,,,,,,,,,,
2015-01-30,AAPL,29.600000,30.000000,29.212500,29.290001,25.973394,334982000,27.660375,29.456897,25.863853,1.814201,0.953550,False,False,True,False,0
2015-02-02,AAPL,29.512501,29.792500,29.020000,29.657499,26.299274,250956400,27.776625,29.773533,25.779717,1.883786,0.970947,False,False,True,False,0
2015-02-03,AAPL,29.625000,29.772499,29.402500,29.662500,26.303719,207662800,27.931625,30.011262,25.851988,1.664594,0.916148,False,False,True,False,0
2015-02-04,AAPL,29.625000,30.127501,29.577499,29.889999,26.505459,280598800,28.097875,30.247931,25.947819,1.667049,0.916762,False,False,True,False,0
2015-02-05,AAPL,30.004999,30.057501,29.812500,29.985001,26.694635,168984800,28.250250,30.484360,26.016140,1.552968,0.888242,False,False,True,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,XOM,114.570000,115.760002,114.470001,114.559998,113.566673,14756100,112.776000,116.029683,109.522317,1.096602,0.774150,False,False,True,False,0
2025-09-25,XOM,114.639999,115.900002,114.410004,115.589996,114.587746,15012600,112.918000,116.406335,109.429665,1.531961,0.882990,False,False,True,False,0
2025-09-26,XOM,115.959999,118.360001,115.919998,117.220001,116.203613,18568900,113.111500,117.094940,109.128060,2.062791,1.015698,False,True,True,False,-1


In [117]:

def generate_risk_metrics(df, n_atr=14, n_vol=20):
    df_risk = df.copy()
    
    #a. ATR
    prev_close = df_risk['Close'].shift(1)
    tr1 = df_risk['High'] - df_risk['Low']
    tr2 = (df_risk['High'] - prev_close).abs()
    tr3 = (df_risk['Low'] - prev_close).abs()
    
    df_risk['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df_risk['ATR'] = df_risk['TR'].rolling(n_atr).mean()
    
    #b. VOLATILITY 
    df_risk['Log_Return'] = np.log(df_risk['Close'] / df_risk['Close'].shift(1))
    df_risk['Vol_Daily'] = df_risk['Log_Return'].rolling(n_vol).std()
    df_risk['Vol_Annual'] = df_risk['Vol_Daily'] * np.sqrt(252)
    

    df_risk['ATR_Prev'] = df_risk['ATR'].shift(1)
    df_risk['Vol_Daily_Prev'] = df_risk['Vol_Daily'].shift(1)
    df_risk['Vol_Annual_Prev'] = df_risk['Vol_Annual'].shift(1)
    
    return df_risk
def full_generate_risk_metrics(df, n_atr = 14, n_vol = 20):
    df_processed = df.groupby("Ticker",group_keys=False).apply(generate_risk_metrics, n_atr=n_atr, n_vol=n_vol)
    return df_processed

In [118]:
processed_stock_70_data=full_generate_risk_metrics(processed_stock_70_data).sort_values(by=['Ticker', 'Date'])
processed_stock_70_data

,,Open,High,Low,Close,Adj Close,Volume,Middle,Upper,Lower,Zscore,PercentB,TR,ATR,Log_Return,Vol_Daily,Vol_Annual,ATR_Prev,Vol_Daily_Prev,Vol_Annual_Prev
Date,Ticker,,,,,,,,,,,,,,,,,,,
2015-01-02,AAPL,27.847500,27.860001,26.837500,27.332500,24.237556,212818400,NaN,NaN,NaN,NaN,NaN,1.022501,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,AAPL,27.072500,27.162500,26.352501,26.562500,23.554747,257142000,NaN,NaN,NaN,NaN,NaN,0.980000,NaN,-0.028576,NaN,NaN,NaN,NaN,NaN
2015-01-06,AAPL,26.635000,26.857500,26.157499,26.565001,23.556952,263188400,NaN,NaN,NaN,NaN,NaN,0.700001,NaN,0.000094,NaN,NaN,NaN,NaN,NaN
2015-01-07,AAPL,26.799999,27.049999,26.674999,26.937500,23.887281,160423600,NaN,NaN,NaN,NaN,NaN,0.484999,NaN,0.013925,NaN,NaN,NaN,NaN,NaN
2015-01-08,AAPL,27.307501,28.037500,27.174999,27.972500,24.805077,237458000,NaN,NaN,NaN,NaN,NaN,1.100000,NaN,0.037703,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,XOM,114.570000,115.760002,114.470001,114.559998,113.566673,14756100,112.7760,116.029683,109.522317,1.096602,0.774150,1.810005,1.966430,0.005339,0.012706,0.201705,1.962143,0.012694,0.201515
2025-09-25,XOM,114.639999,115.900002,114.410004,115.589996,114.587746,15012600,112.9180,116.406335,109.429665,1.531961,0.882990,1.489998,1.831429,0.008951,0.012623,0.200378,1.966430,0.012706,0.201705
2025-09-26,XOM,115.959999,118.360001,115.919998,117.220001,116.203613,18568900,113.1115,117.094940,109.128060,2.062791,1.015698,2.770004,1.907858,0.014003,0.012916,0.205040,1.831429,0.012623,0.200378


In [119]:
#3. Chia train,test 

def split_walk_forward_rolling(df, train_window, test_size):
    total_rows = len(df)
    
    for i in range(train_window, total_rows, test_size):
        train_end = i
        test_end = min(i + test_size, total_rows)
        
        # Điểm bắt đầu của Train trượt theo i
        train_start = i - train_window
        
        if test_end <= train_end:
            break
            
        train_data = df.iloc[train_start:train_end].copy()
        test_data = df.iloc[train_end:test_end].copy()
        
        yield train_data, test_data

def split_walk_forward_expanding(df, train_size, test_size):

    total_rows = len(df)

    for i in range(train_size, total_rows, test_size):
        train_end = i
        test_end = min(i + test_size, total_rows)
        
        if test_end <= train_end:
            break
            
        train_data = df.iloc[:train_end].copy()
        test_data = df.iloc[train_end:test_end].copy()

        yield train_data, test_data

def split_simple_train_test(df, train_ratio=0.8):
   
    n = len(df)
    split_index = int(n * train_ratio)
    
    train_data = df.iloc[:split_index].copy()
    test_data = df.iloc[split_index:].copy()
    
    return train_data, test_data

In [120]:
def calculate_position_size(account_size, risk_pct, vol_annual, price, m=2.0):

    risk_amount = account_size * risk_pct
    vol_impact = (vol_annual * m * price) + 1e-9
    
    # Tính khối lượng
    size = risk_amount / vol_impact

    return int(size)

In [121]:

def calculate_initial_stop_loss(entry_price, method ='sigma', vol_daily= None, atr = None, k_vol = 2, k_atr = 3, k_fixed = 0.01, position_type=1):
    stop_loss_price = 0
    distance = 0
    
    # 1. TÍNH KHOẢNG CÁCH (MAGNITUDE) - Phần này giống nhau cả 2 phe
    if method == 'sigma':
        distance = k_vol * vol_daily

    elif method == 'atr':
        distance = k_atr * atr

    elif method == 'fixed_pct':
        distance = entry_price * k_fixed
  
    if position_type == 1:  # LONG
        stop_loss_price = entry_price - distance
        
    elif position_type == -1: # SHORT
        stop_loss_price = entry_price + distance
        
    return stop_loss_price

In [ ]:

def calculate_trade_metrics(trade_log, total_bars=252):
    """
    Tính toán 11 chỉ số hiệu suất cấp độ Lệnh (Trade-Level) đầy đủ.
    
    Tham số:
    - trade_log: List các dict kết quả lệnh [{'pnl':.., 'return':.., 'bars':..}, ...]
    - total_bars: Tổng số nến (ngày) của giai đoạn backtest (để tính Time in Market).
    """
    
    df = pd.DataFrame(trade_log)
    
    if df.empty:
        return {"Status": "Không có giao dịch nào"}
        
    # Chuẩn bị 
    winning_trades = df[df['pnl'] > 0]
    losing_trades = df[df['pnl'] <= 0]
    
    n_win = len(winning_trades)
    n_loss = len(losing_trades)
    n_total = n_win + n_loss
    
    # Tính toán các metric 
    gross_gain = winning_trades['pnl'].sum()
    gross_loss = losing_trades['pnl'].abs().sum()
    net_profit = gross_gain - gross_loss
    total_days_in_market = df['bars'].sum()
    time_in_market = total_days_in_market / total_bars
    number_of_trades = n_total
    years = total_bars / 252
    annual_turnover = n_total / years if years > 0 else n_total
    hit_rate = n_win / n_total if n_total > 0 else 0
    profit_factor = gross_gain / gross_loss if gross_loss != 0 else np.inf
    avg_gain = winning_trades['return'].mean() if n_win > 0 else 0
    avg_loss = abs(losing_trades['return'].mean()) if n_loss > 0 else 0
    slugging_ratio = avg_gain / avg_loss if avg_loss != 0 else np.inf

    # --- 3. TRẢ VỀ KẾT QUẢ ---
    metrics = {
        "1. Gross Gain": gross_gain,
        "2. Gross Loss": gross_loss,
        "3. Net Profit": net_profit,
        "4. % Time in Market": f"{time_in_market:.2%}",
        "5. Number of Trades": number_of_trades,
        "6. Annual Turnover": round(annual_turnover, 2),
        "7. Hit Rate": f"{hit_rate:.2%}",
        "8. Profit Factor": round(profit_factor, 2),
        "9. Avg Gain (Wins)": f"{avg_gain:.2%}",
        "10. Avg Loss (Losses)": f"{avg_loss:.2%}",
        "11. Slugging Ratio": round(slugging_ratio, 2)
    }
    
    return metrics

def get_performance_trade_metrics(rolling_results, total_bars_in_data):

    all_trades_list = []
    
    for period in rolling_results:
        df_trades = period['trades']
        
        for _, row in df_trades.iterrows():
            all_trades_list.append({
                'pnl': row['PnL'],
                'return': row['ReturnPct'],
                'bars': row['ExitBar'] - row['EntryBar']
            })
    
    metrics = calculate_trade_metrics(all_trades_list, total_bars=total_bars_in_data)
    
    return metrics

def calculate_single_ticker_metrics(ticker_res, total_test_bars):
    
    df_trades = ticker_res['trades']
    
    current_ticker_trades = []
    for _, row in df_trades.iterrows():
        current_ticker_trades.append({
            'pnl': row['PnL'],
            'return': row['ReturnPct'],
            'bars': row['ExitBar'] - row['EntryBar']
        })
    
    metrics = calculate_trade_metrics(current_ticker_trades, total_bars=total_test_bars)
    
    return metrics

def calculate_portfolio_trading_metrics(all_ticker_results, total_test_bars):
    portfolio_trades = []
    for res in all_ticker_results:
        df_trades = res['trades']
        for _, row in df_trades.iterrows():
            portfolio_trades.append({
                'pnl': row['PnL'],
                'return': row['ReturnPct'],
                'bars': row['ExitBar'] - row['EntryBar']
            })
    
    # total_bars ở đây được hiểu là độ dài thời gian quan sát của danh mục
    portfolio_metrics = calculate_trade_metrics(portfolio_trades, total_bars=total_test_bars)

    portfolio_metrics['Total Tickers'] = len(all_ticker_results)
    portfolio_metrics['Total Portfolio Trades'] = len(portfolio_trades)
    
    return portfolio_metrics


In [181]:


class Bollinger_Mean_Reversion(Strategy):
    # Khai báo tham số để Optimize (Thư viện sẽ dùng các biến này)
    n_param = 20
    k_param = 2
    risk_pct = 0.02
    k_vol_sl = 2
    k_atr = 3
    k_fixed = 0.01
    sl_method = 'atr'

    def init(self):
        df_raw = self.data.df 
       
        def get_indicators(df):
            d = generate_bollinger_band_indicator(df, N=self.n_param, K=self.k_param)
            d = generate_risk_metrics(d)
            d = generate_bollinger_signal(d) 
            return d

        processed = get_indicators(df_raw)
        
        self.sig = self.I(lambda: processed['Signal'], name='Signal')
        self.mid = self.I(lambda: processed['Middle'], name='Middle')
        self.upper = self.I(lambda: processed['Upper'], name='Upper')
        self.lower = self.I(lambda: processed['Lower'], name='Lower')
        self.v_daily = self.I(lambda: processed['Vol_Daily'], name='Vol_Daily')
        self.v_annual = self.I(lambda: processed['Vol_Annual'], name='Vol_Annual')
        self.atr_val = self.I(lambda: processed['ATR'], name='ATR')
 

    def next(self):
        price = self.data.Close[-1]
        signal = self.sig[-1]

        # --- LOGIC VÀO LỆNH (Tối nay có signal, sáng mai khớp) ---
        if not self.position:
            # LONG
            if signal == 1:
                sl = calculate_initial_stop_loss(price, self.sl_method, self.v_daily[-1], self.atr_val[-1], self.k_vol_sl, self.k_atr, self.k_fixed, 1)
                size = calculate_position_size(self.equity, self.risk_pct, self.v_annual[-1], price)
                self.buy(size=size, sl=sl)

            # SHORT
            elif signal == -1:
                sl = calculate_initial_stop_loss(price, self.sl_method, self.v_daily[-1], self.atr_val[-1], self.k_vol_sl, self.k_atr, self.k_fixed, -1)
                size = calculate_position_size(self.equity, self.risk_pct, self.v_annual[-1], price)
                self.sell(size=size, sl=sl)

        # --- LOGIC THOÁT LỆNH (Mean Reversion) ---
        elif self.position.is_long and signal == 0:
            self.position.close() # Mua trả hàng / Bán chốt lời vào sáng mai
            
        elif self.position.is_short and signal == 0:
            self.position.close()

In [ ]:
def run_walk_forward_backtest(df_input, train_window=252*5, test_size=252):

    all_stats = []

    for train_data, test_data in split_walk_forward_rolling(df_input, train_window, test_size):
        
        print(f"--- Đang xử lý giai đoạn: {test_data.index[0].date()} đến {test_data.index[-1].date()} ---")
        
        bt_train = Backtest(train_data, Bollinger_Mean_Reversion, cash=1_000_000_000, commission=0.0015, trade_on_close=True)
        

        train_results = bt_train.optimize(
            n_param= 20,
            k_param= 2,
            maximize='Return [%]'
        )
        
        best_n = train_results._strategy.n_param
        best_k = train_results._strategy.k_param
        
        class BestParamsStrategy(Bollinger_Mean_Reversion):
            n_param = best_n
            k_param = best_k

        bt_test = Backtest(test_data, BestParamsStrategy, cash=1_000_000_000, commission=0.0015, trade_on_close=True)
        test_stats = bt_test.run()
        
        all_stats.append({
            'period_start': test_data.index[0],
            'period_end': test_data.index[-1],
            'best_n': best_n,
            'best_k': best_k,
            'return': test_stats['Return [%]'],
            'win_rate': test_stats['Win Rate [%]'],
            'trades': test_stats['_trades'] 
        })

    return all_stats



In [195]:
def run_simple_backtest(df_input, train_ratio = 0.7):
    all_stats = []
    train_data, test_data = split_simple_train_test(df_input,train_ratio= train_ratio)

    bt_train = Backtest(train_data, Bollinger_Mean_Reversion, cash=1_000_000_000, commission=0.0015, trade_on_close=True)
    train_results = bt_train.optimize(
        n_param = 20,
        k_param = 2,
        maximize = 'Return [%]'
    )

    best_n = train_results._strategy.n_param
    best_k = train_results._strategy.k_param

    class BestParamsStrategy(Bollinger_Mean_Reversion):
            n_param = best_n
            k_param = best_k

    bt_test = Backtest(test_data, BestParamsStrategy, cash=1_000_000_000, commission=0.0015, trade_on_close=True)
    test_stats = bt_test.run()

    all_stats.append({
            'period_start': test_data.index[0],
            'period_end': test_data.index[-1],
            'best_n': best_n,
            'best_k': best_k,
            'return': test_stats['Return [%]'],
            'win_rate': test_stats['Win Rate [%]'],
            'trades': test_stats['_trades'] ,
           # 'equity_curve': test_stats['_equity_curve']
        })
    return all_stats

   


In [ ]:


def full_run_simple_backtest(all_stock_data, train_ratio=0.7):

    tickers = all_stock_data.index.get_level_values(1).unique()
    final_results_list = []
    
    print(f" Bắt đầu quét danh mục {len(tickers)} mã...")

    for i, ticker in enumerate(tickers):
        try:
           
            df_ticker = all_stock_data.xs(ticker, level=1).sort_index()

            ticker_stats_list = run_simple_backtest(df_ticker, train_ratio=train_ratio)
            
            if ticker_stats_list:
                res = ticker_stats_list[0]
                df_trades = res['trades']
                
        
                total_test_bars = len(df_ticker.iloc[int(len(df_ticker) * train_ratio):])
            
                all_trades = []
                for _, row in df_trades.iterrows():
                    all_trades.append({
                        'pnl': row['PnL'],
                        'return': row['ReturnPct'],
                        'bars': row['ExitBar'] - row['EntryBar']
                    })
                
    
                ticker_metrics = calculate_trade_metrics(all_trades, total_bars=total_test_bars)
                

                ticker_metrics['Ticker'] = ticker
                final_results_list.append(ticker_metrics)
                
                print(f"[{i+1}/{len(tickers)}] ✅ {ticker} | Net Profit: {ticker_metrics.get('3. Net Profit', 0):,.0f}")
            
        except Exception as e:
            print(f"[{i+1}/{len(tickers)}] ❌ Lỗi tại mã {ticker}: {e}")

    # 4. Tổng hợp thành bảng báo cáo duy nhất
    df_final_report = pd.DataFrame(final_results_list)


    if not df_final_report.empty:
        cols = ['Ticker'] + [c for c in df_final_report.columns if c != 'Ticker']
        df_final_report = df_final_report[cols]

        if '3. Net Profit' in df_final_report.columns:
            df_final_report = df_final_report.sort_values(by="3. Net Profit", ascending=False)

    print("\n--- QUÁ TRÌNH HOÀN TẤT ---")
    return df_final_report


report_70_ma = full_run_simple_backtest(stock_70_data)

In [193]:
report_70_ma

,Ticker,1. Gross Gain,2. Gross Loss,3. Net Profit,4. % Time in Market,5. Number of Trades,6. Annual Turnover,7. Hit Rate,8. Profit Factor,9. Avg Gain (Wins),10. Avg Loss (Losses),11. Slugging Ratio
61,SRE,5.187787e+07,2.714002e+07,2.473785e+07,42.66%,38,11.81,71.05%,1.91,4.08%,5.26%,0.77
69,XOM,3.937361e+07,1.500952e+07,2.436409e+07,37.24%,33,10.25,78.79%,2.62,3.43%,4.19%,0.82
18,CVX,3.286441e+07,1.453939e+07,1.832502e+07,42.91%,32,9.94,75.00%,2.26,2.95%,4.03%,0.73
15,COP,4.334756e+07,2.509191e+07,1.825564e+07,41.68%,36,11.19,72.22%,1.73,4.49%,5.73%,0.78
23,EOG,4.101240e+07,2.412620e+07,1.688620e+07,45.99%,39,12.12,69.23%,1.70,4.15%,5.81%,0.71
...,...,...,...,...,...,...,...,...,...,...,...,...
62,TMO,2.381590e+07,5.121569e+07,-2.739979e+07,53.14%,39,12.12,53.85%,0.47,2.33%,6.02%,0.39
19,DIS,2.693651e+07,5.806508e+07,-3.112857e+07,54.75%,41,12.74,48.78%,0.46,4.03%,6.89%,0.58
33,KO,3.043089e+07,6.190318e+07,-3.147229e+07,45.62%,39,12.12,48.72%,0.49,2.20%,4.21%,0.52
26,GE,2.524283e+07,6.094067e+07,-3.569783e+07,49.57%,37,11.50,40.54%,0.41,4.94%,7.03%,0.70


In [ ]:


# 1. Lấy danh sách tất cả các mã cổ phiếu
all_tickers = loader.TICKERS_70

final_results_list = []

print(f"Bắt đầu quét {len(all_tickers)} mã. Chấp nhận sai số metrics thời gian...")

for ticker in all_tickers:
    try:
        # Lấy dữ liệu mã đó
        df_ticker = stock_70_data.xs(ticker, level=1).sort_index()
        
        # Chạy Rolling Backtest
        rolling_res =  run_walk_forward_backtest(df_ticker)
        
        if rolling_res:
            # GOM TẤT CẢ LỆNH (Chấp nhận trùng do chồng lấn nến)
            all_trades = []
            total_bars_sum = 0
            
            for period in rolling_res:
                total_bars_sum += 60  # Giả sử mỗi đoạn test là 60 nến
                for _, row in period['trades'].iterrows():
                    all_trades.append({
                        'pnl': row['PnL'],
                        'return': row['ReturnPct'],
                        'bars': row['ExitBar'] - row['EntryBar']
                    })
            
            # 2. Tính Metrics bằng hàm của bạn
            ticker_metrics = calculate_trade_metrics(all_trades, total_bars=total_bars_sum)
            
            # Thêm tên mã để dễ nhận biết
            ticker_metrics['Ticker'] = ticker
            final_results_list.append(ticker_metrics)
            
            print(f"✅ Xong {ticker} | Net Profit: {ticker_metrics['3. Net Profit']:,.0f}")
            
    except Exception as e:
        print(f"❌ Lỗi tại mã {ticker}: {e}")

# 3. Tổng hợp thành bảng xếp hạng
df_final_report = pd.DataFrame(final_results_list)

# Đưa cột Ticker lên đầu cho dễ nhìn
cols = ['Ticker'] + [c for c in df_final_report.columns if c != 'Ticker']
df_final_report = df_final_report[cols]

# Sắp xếp theo Net Profit để xem mã nào ra tiền nhiều nhất
leaderboard = df_final_report.sort_values(by="3. Net Profit", ascending=False)

In [ ]:

import os 
def plot_trading_signal(df_input, ticker_name, best_n, best_k, train_ratio=0.7, output_dir = 'result'):
    """
    Hàm vẽ đồ thị kỹ thuật (Candlestick + Bollinger Bands + Signals) cho 1 mã duy nhất.
    """
    try:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        # Trích xuất dữ liệu của mã đó
        df_ticker = df_input.xs(ticker_name, level=1).sort_index()
        
        # Truyền các tham số 
        class FinalStrategy(Bollinger_Mean_Reversion):
            n_param = best_n
            k_param = best_k

        _, test_data = split_simple_train_test(df_ticker, train_ratio =train_ratio)
        bt_plot = Backtest(test_data, FinalStrategy, 
                           cash=1_000_000_000, 
                           commission=0.0015, 
                           trade_on_close=True)
        
      
        bt_plot.run()
        
        file_name = f"{ticker_name}_N{best_n}_K{best_k}.html"
        file_path = os.path.join(output_dir, file_name)
        
        print(f"Đang vẽ đồ thị mã: {ticker_name}")
        print(f"File HTML sẽ được lưu tại: {file_path}")

        # 7. Lệnh vẽ và lưu file
        # filename: tên file lưu | open_browser: True để tự động mở tab mới
        return bt_plot.plot(filename=file_path, open_browser=True)
        
    except Exception as e:
        print(f"Lỗi khi vẽ mã {ticker_name}: {e}")
